# Kémzy àvátâr — PersonaLive CUDA proof v2

This version fixes the previous Kaggle failure by treating the PersonaLive SHA as a commit, not a branch. It checks out the exact upstream commit, uses the upstream Python 3.10 environment, verifies the official weight manifest, then runs the real Kémzy PersonaLive render gateway.

In [ ]:
import os, subprocess, shutil, time, base64, requests
ROOT='/kaggle/working'
KEMZY=f'{ROOT}/Kemzy-LiveAvatar'
PERSONA=f'{ROOT}/PersonaLive'
PIN='abdd112e01dcf7d89122c2e5efa29fcff0669740'
subprocess.run(['rm','-rf',KEMZY,PERSONA],check=True)
subprocess.run(['git','clone','--depth','1','--branch','feature/backend-render-gateway','https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git',KEMZY],check=True)
subprocess.run(['git','clone','https://github.com/GVCLab/PersonaLive.git',PERSONA],check=True)
subprocess.run(['git','-C',PERSONA,'checkout','--detach',PIN],check=True)
actual=subprocess.check_output(['git','-C',PERSONA,'rev-parse','HEAD'],text=True).strip()
print('PersonaLive commit:',actual)
assert actual==PIN, (actual,PIN)
assert os.path.isfile(f'{KEMZY}/backend/renderer/personalive_server.py')
print('CUDA:')
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
# PersonaLive upstream specifies Python 3.10; Kaggle's notebook kernel is newer.
conda=shutil.which('conda') or '/opt/conda/bin/conda'
assert os.path.exists(conda), conda
subprocess.run([conda,'create','-y','-n','personalive','python=3.10'],check=True)
subprocess.run([conda,'run','-n','personalive','python','-m','pip','install','-r',f'{PERSONA}/requirements_base.txt'],check=True)
print('Pinned PersonaLive environment ready.')


In [ ]:
required_repo=['requirements_base.txt','tools/download_weights.py','configs/prompts/personalive_online.yaml','webcam/vid2vid.py','src/wrapper.py']
missing=[p for p in required_repo if not os.path.isfile(os.path.join(PERSONA,p))]
assert not missing,missing
subprocess.run([conda,'run','-n','personalive','python','tools/download_weights.py'],cwd=PERSONA,check=True)
required_weights=[
 'pretrained_weights/personalive/denoising_unet.pth',
 'pretrained_weights/personalive/motion_encoder.pth',
 'pretrained_weights/personalive/motion_extractor.pth',
 'pretrained_weights/personalive/pose_guider.pth',
 'pretrained_weights/personalive/reference_unet.pth',
 'pretrained_weights/personalive/temporal_module.pth',
 'pretrained_weights/sd-vae-ft-mse/diffusion_pytorch_model.bin',
 'pretrained_weights/sd-vae-ft-mse/config.json',
 'pretrained_weights/sd-image-variations-diffusers/image_encoder/pytorch_model.bin',
 'pretrained_weights/sd-image-variations-diffusers/image_encoder/config.json',
 'pretrained_weights/sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin',
 'pretrained_weights/sd-image-variations-diffusers/unet/config.json',
 'pretrained_weights/sd-image-variations-diffusers/model_index.json']
missing=[p for p in required_weights if not os.path.isfile(os.path.join(PERSONA,p))]
print('Missing weights:',missing)
assert not missing,missing
print('Official weight manifest: PASS')


## Upload reference portrait and optional driving video


In [ ]:
from IPython.display import display
from ipywidgets import FileUpload
reference_upload=FileUpload(accept='image/*',multiple=False)
video_upload=FileUpload(accept='video/*',multiple=False)
display(reference_upload)
display(video_upload)


In [ ]:
from PIL import Image
import io
assert reference_upload.value,'Upload a reference portrait first.'
item=next(iter(reference_upload.value.values()))
Image.open(io.BytesIO(item['content'])).convert('RGB').save(f'{ROOT}/reference.jpg',quality=95)
if video_upload.value:
 v=next(iter(video_upload.value.values()))
 open(f'{ROOT}/driving.mp4','wb').write(v['content'])
 subprocess.run(['ffmpeg','-y','-i',f'{ROOT}/driving.mp4','-vf','select=not(mod(n\,4))','-frames:v','4',f'{ROOT}/frame%01d.jpg'],check=True)
else:
 img=Image.open(f'{ROOT}/reference.jpg').convert('RGB')
 [img.save(f'{ROOT}/frame{i}.jpg',quality=92) for i in range(1,5)]
 print('No driving video supplied; using reference frames as a smoke-test fallback.')
assert all(os.path.isfile(f'{ROOT}/frame{i}.jpg') for i in range(1,5))
print('Reference bytes:',os.path.getsize(f'{ROOT}/reference.jpg'))
print('Driving frames ready.')


In [ ]:
# Copy the Kémzy gateway into the pinned PersonaLive checkout.
server_src=f'{KEMZY}/backend/renderer/personalive_server.py'
server_dst=f'{PERSONA}/personalive_server.py'
shutil.copy2(server_src,server_dst)
# The upstream README/downloader use unet/diffusion_pytorch_model.bin; the old gateway
# incorrectly required an additional unet/pytorch_model.bin. Remove only that stale check.
text=open(server_dst,encoding='utf-8').read()
stale='    "sd-image-variations-diffusers/unet/pytorch_model.bin",\n'
assert stale in text,'stale weight check was not found; refusing to hide a mismatch'
text=text.replace(stale,'')
open(server_dst,'w',encoding='utf-8').write(text)
print('Gateway prepared:',server_dst)
probe=subprocess.run([conda,'run','-n','personalive','python','-c','import webcam.vid2vid,src.wrapper; print(webcam.vid2vid.Pipeline.__module__); print(src.wrapper.PersonaLive.__module__)'],cwd=PERSONA,capture_output=True,text=True)
print(probe.stdout)
print(probe.stderr)
probe.check_returncode()


In [ ]:
env=os.environ.copy()
env.update({'MODEL_DIR':PERSONA,'PERSONALIVE_CONFIG':f'{PERSONA}/configs/prompts/personalive_online.yaml','ACCELERATION':'none','DIAGNOSTIC_RENDER_TIMEOUT':'180','PERSONALIVE_COMMIT':PIN,'PYTHONUNBUFFERED':'1'})
server=subprocess.Popen([conda,'run','--no-capture-output','-n','personalive','python','-m','uvicorn','personalive_server:app','--host','0.0.0.0','--port','7860'],cwd=PERSONA,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
time.sleep(15)
health=requests.get('http://127.0.0.1:7860/health',timeout=30)
print('Health HTTP:',health.status_code)
print(health.text)
health.raise_for_status()
state=health.json()
assert state['cuda'] is True,state
assert state['weights_ready'] is True,state
print('Renderer health: PASS')


In [ ]:
paths={'reference':f'{ROOT}/reference.jpg','frame1':f'{ROOT}/frame1.jpg','frame2':f'{ROOT}/frame2.jpg','frame3':f'{ROOT}/frame3.jpg','frame4':f'{ROOT}/frame4.jpg'}
handles=[]
files={}
try:
 for field,path in paths.items():
  f=open(path,'rb'); handles.append(f); files[field]=(os.path.basename(path),f,'image/jpeg')
 r=requests.post('http://127.0.0.1:7860/v1/diagnostics/render',files=files,timeout=300)
 print('Render HTTP:',r.status_code)
 print(r.text[:5000])
 r.raise_for_status()
 result=r.json()
 assert result['status']=='rendered',result
 assert result['generated_frames']>0,result
 jpeg=base64.b64decode(result['first_frame_jpeg_base64'])
 open(f'{ROOT}/kemzy_personalive_render.jpg','wb').write(jpeg)
 print('VERIFIED generated JPEG bytes:',len(jpeg))
 print('Render seconds:',result['render_seconds'])
finally:
 for f in handles: f.close()


In [ ]:
from IPython.display import display
out=f'{ROOT}/kemzy_personalive_render.jpg'
assert os.path.isfile(out) and os.path.getsize(out)>1000
display(Image.open(out))
print('FINAL PROOF: CUDA + exact PersonaLive commit + official weights + generated frame = PASS')
